# Regression Target Engineering: Full Triage (Z-Score, Fraction, Log)

## 0. Setup

In [1]:
import csv
import random
import re
import json
import math
import time
import traceback
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torchvision.transforms.functional as TF
from torchvision import transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from torchvision.models import convnext_tiny, ConvNeXt_Tiny_Weights
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.utils.checkpoint import checkpoint
from sklearn.metrics import (f1_score, roc_auc_score, mean_absolute_error, mean_squared_error,
                              r2_score)

MANIFEST = Path('~/Desktop/convscript/Code/Manifest/manifests/master_manifest.csv').expanduser()
ROOT     = Path('~/Desktop/Thesis/TR-6').expanduser()
GAS_NORM = Path('~/Desktop/convscript/Code/Manifest/GasNorm/gas_norm_stats.json').expanduser()
RUNS     = Path('~/Desktop/convscript/runs').expanduser()

DEVICE = (
    'mps'  if torch.backends.mps.is_available() else
    'cuda' if torch.cuda.is_available()          else
    'cpu'
)
print(f'Device: {DEVICE}')

OUT_DIR = RUNS / "13_regression_target_triage"
OUT_DIR.mkdir(parents=True, exist_ok=True)

FAILURES = []
THRESHOLDS_TO_SWEEP = [t / 100 for t in range(10, 91)]
N_SEEDS = 5

FULL_MODEL_VAL_F1_KNOWN = 0.8000
FULL_MODEL_TEST_RESULT = {"test_f1": 0.8000, "test_auc": 0.7333, "test_mae": 0.9188,
                           "test_rmse": 1.0593, "test_r2": 0.0439}


Device: mps


## Data pipeline

In [2]:
FRUIT_LIST   = ['Banana', 'Carrot', 'Guava', 'Indian_Gooseberry', 'Mango', 'Tomato']
FRUIT_TO_IDX = {f: i for i, f in enumerate(FRUIT_LIST)}
SESSION_TO_IDX = {'morning': 0, 'afternoon': 1, 'evening': 2}
LABEL_TO_IDX   = {'not_spoiled': 0, 'spoiled': 1}
IMAGE_SIZE     = 224
IMAGENET_MEAN  = [0.485, 0.456, 0.406]
IMAGENET_STD   = [0.229, 0.224, 0.225]
SESSION_HOUR_BUCKETS = {'morning': (5, 11), 'afternoon': (12, 16), 'evening': (16, 19)}
IR_PATTERN   = re.compile(r'(\d{8})_(\d{6})_([\d.]+)C_([\d.]+)C\.jpg$', re.IGNORECASE)
SRGB_PATTERN = re.compile(r'^(\d{8})_(\d{6})[^/]*\.jpg$', re.IGNORECASE)


def safe_float(val, default=0.0):
    try:
        return float(val)
    except Exception:
        return default


def hour_to_session(hour):
    for name, (lo, hi) in SESSION_HOUR_BUCKETS.items():
        if lo <= hour < hi:
            return name
    return 'unknown'


def find_ir_folder(base):
    for name in ['IR_fusion_images', 'IR_Fusion_images', 'ir_fusion_images']:
        p = base / name
        if p.exists():
            return p
    return base / 'IR_fusion_images'


def group_images_by_session(folder, pattern):
    result = defaultdict(lambda: defaultdict(list))
    if not folder.exists():
        return {}
    for f in sorted(folder.iterdir()):
        m = pattern.search(f.name)
        if not m:
            continue
        session = hour_to_session(int(m.group(2)[:2]))
        if session != 'unknown':
            result[m.group(1)][session].append(f)
    return dict(result)


def load_image(path, transform):
    return transform(Image.open(path).convert('RGB'))


def get_rgb_transform(train: bool):
    if train:
        return transforms.Compose([
            transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
            transforms.RandomHorizontalFlip(), transforms.RandomRotation(15),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
            transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ])
    return transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])


def get_ir_transform(train: bool):
    if train:
        return transforms.Compose([
            transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
            transforms.RandomHorizontalFlip(), transforms.RandomRotation(15),
            transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ])
    return transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])


class TrimodalDataset(Dataset):
    def __init__(self, manifest_path, root, train=True, modality_dropout_prob=0.3,
                 exclude_flagged=True, split=None, gas_norm_stats_path=None,
                 cache_images=True, shared_pixel_cache=None):
        self.root = Path(root); self.train = train; self.modality_dropout_prob = modality_dropout_prob
        self.rgb_transform = get_rgb_transform(train); self.ir_transform = get_ir_transform(train)
        self.gas_norm_stats = None
        if gas_norm_stats_path:
            with open(gas_norm_stats_path) as f:
                self.gas_norm_stats = json.load(f)['stats']
        with open(manifest_path, newline='') as f:
            all_rows = list(csv.DictReader(f))
        if exclude_flagged:
            all_rows = [r for r in all_rows if r.get('exclude', '').strip().lower() != 'true']
        if split:
            all_rows = [r for r in all_rows if r.get('split', '').strip().lower() == split.lower()]
        self.samples = []; self._build_samples(all_rows)
        self._image_cache = {}; self._build_image_index()
        self._pixel_cache = {}
        if shared_pixel_cache is not None:
            self._pixel_cache = shared_pixel_cache
        elif cache_images:
            self._warmup_pixel_cache()
        print(f'TrimodalDataset: {len(self.samples)} sessions (split={split})')

    def _build_samples(self, rows):
        for row in rows:
            fruit = row['fruit']; label = row['label']; date_str = row['actual_date']
            global_day = int(row['corrected_day_index'])
            days_until = safe_float(row.get('days_until_spoilage', ''), default=-1.0)
            for session in ['morning', 'afternoon', 'evening']:
                si = SESSION_TO_IDX[session]
                ir_avail = safe_float(row.get(f'ir_{session}_count', 0)) > 0
                srgb_avail = safe_float(row.get(f'srgb_{session}_count', 0)) > 0
                gas_avail = row.get(f'methane_{session}_present', '').strip().upper() == 'TRUE'
                mean_ppm = safe_float(row.get(f'methane_{session}_ppm', ''), 0.0)
                std_ppm = safe_float(row.get(f'methane_{session}_std', ''), 0.0)
                left_ppm = safe_float(row.get(f'methane_{session}_left', ''), 0.0)
                right_ppm = safe_float(row.get(f'methane_{session}_right', ''), 0.0)
                if not ir_avail and not srgb_avail and not gas_avail:
                    continue
                self.samples.append({'fruit': fruit, 'label': label, 'actual_date': date_str,
                    'corrected_day_index': global_day, 'session': session, 'session_idx': si,
                    'fruit_idx': FRUIT_TO_IDX.get(fruit, 0), 'label_idx': LABEL_TO_IDX.get(label, 0),
                    'days_until_spoilage': days_until, 'gas_mean': mean_ppm, 'gas_std': std_ppm,
                    'gas_left': left_ppm, 'gas_right': right_ppm,
                    'rgb_available': srgb_avail, 'ir_available': ir_avail, 'gas_available': gas_avail})

    def _build_image_index(self):
        label_map = {'not_spoiled': 'Not_spoiled', 'spoiled': 'Spoiled'}
        for fruit in FRUIT_LIST:
            for lk, lf in label_map.items():
                base = self.root / 'Classified' / fruit / lf
                for d, ss in group_images_by_session(base / 'sRGB_images', SRGB_PATTERN).items():
                    for s, ps in ss.items():
                        self._image_cache.setdefault((fruit, lk, d, s), {'rgb': [], 'ir': []})['rgb'] = ps
                for d, ss in group_images_by_session(find_ir_folder(base), IR_PATTERN).items():
                    for s, ps in ss.items():
                        self._image_cache.setdefault((fruit, lk, d, s), {'rgb': [], 'ir': []})['ir'] = ps
        banana_ir = group_images_by_session(find_ir_folder(self.root / 'Normal' / 'Banana'), IR_PATTERN)
        spoiled_dates = {date for (fr, lb, date, _) in self._image_cache if fr == 'Banana' and lb == 'spoiled'}
        for d, ss in banana_ir.items():
            if d in spoiled_dates:
                for s, ps in ss.items():
                    key = ('Banana', 'spoiled', d, s)
                    self._image_cache.setdefault(key, {'rgb': [], 'ir': []})
                    if not self._image_cache[key].get('ir'):
                        self._image_cache[key]['ir'] = ps

    def _load_cached(self, path, transform):
        cached = self._pixel_cache.get(str(path))
        if cached is not None:
            return transform(Image.fromarray(cached.permute(1, 2, 0).numpy()))
        return load_image(path, transform)

    def _warmup_pixel_cache(self):
        all_paths = set()
        for v in self._image_cache.values():
            all_paths.update(v.get('rgb', [])); all_paths.update(v.get('ir', []))
        for path in all_paths:
            try:
                img = self.rgb_transform.transforms[0](Image.open(path).convert('RGB'))
                self._pixel_cache[str(path)] = torch.from_numpy(np.array(img)).permute(2, 0, 1)
            except Exception:
                pass

    def _apply_dropout(self, rgb_avail, ir_avail, gas_avail):
        if not self.train:
            return rgb_avail, ir_avail, gas_avail
        while True:
            r = rgb_avail and (random.random() > self.modality_dropout_prob)
            i = ir_avail and (random.random() > self.modality_dropout_prob)
            g = gas_avail and (random.random() > self.modality_dropout_prob)
            if r or i or g:
                return r, i, g

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        fruit = s['fruit']; label = s['label']; date_str = s['actual_date']; session = s['session']
        rgb_avail, ir_avail, gas_avail = self._apply_dropout(s['rgb_available'], s['ir_available'], s['gas_available'])
        cached = self._image_cache.get((fruit, label, date_str, session), {'rgb': [], 'ir': []})
        if rgb_avail and cached['rgb']:
            rgb_tensors = torch.stack([self._load_cached(p, self.rgb_transform) for p in cached['rgb']])
        else:
            rgb_avail = False; rgb_tensors = torch.zeros(1, 3, IMAGE_SIZE, IMAGE_SIZE)
        if ir_avail and cached['ir']:
            ir_tensors = torch.stack([self._load_cached(p, self.ir_transform) for p in cached['ir']])
        else:
            ir_avail = False; ir_tensors = torch.zeros(1, 3, IMAGE_SIZE, IMAGE_SIZE)
        if gas_avail:
            gas = torch.tensor([s['gas_mean'], s['gas_std'], s['gas_left'], s['gas_right'],
                                 abs(s['gas_left'] - s['gas_right']), float(s['session_idx']) / 2.], dtype=torch.float32)
        else:
            gas = torch.zeros(6)
        return {'rgb_images': rgb_tensors, 'ir_images': ir_tensors, 'gas': gas,
                'rgb_available': torch.tensor(rgb_avail, dtype=torch.bool),
                'ir_available': torch.tensor(ir_avail, dtype=torch.bool),
                'gas_available': torch.tensor(gas_avail, dtype=torch.bool),
                'fruit_idx': torch.tensor(s['fruit_idx'], dtype=torch.long),
                'session_idx': torch.tensor(s['session_idx'], dtype=torch.long),
                'label': torch.tensor(s['label_idx'], dtype=torch.long),
                'days_until_spoilage': torch.tensor(s['days_until_spoilage'], dtype=torch.float32),
                'fruit': fruit, 'actual_date': date_str, 'corrected_day_index': s['corrected_day_index']}


class DayLevelSequenceDataset(Dataset):
    def __init__(self, manifest_path=MANIFEST, root=ROOT, split="train", gas_norm_stats_path=GAS_NORM,
                 shared_pixel_cache=None):
        base = TrimodalDataset(manifest_path, root, train=False, modality_dropout_prob=0.0, split=split,
                                gas_norm_stats_path=gas_norm_stats_path, cache_images=True,
                                shared_pixel_cache=shared_pixel_cache)
        self._pixel_cache = base._pixel_cache
        day_groups = defaultdict(list)
        for i in range(len(base)):
            item = base[i]
            key = (item["fruit"], int(item["label"].item()), int(item["corrected_day_index"]))
            day_groups[key].append(item)
        day_entries = {}
        for (fruit, label, day_idx), sessions in day_groups.items():
            sessions.sort(key=lambda s: int(s["session_idx"].item()))
            rgb_img, ir_img = None, None
            for s in sessions:
                if rgb_img is None and bool(s["rgb_available"].item()) and len(s["rgb_images"]) > 0:
                    rgb_img = s["rgb_images"][0]
                if ir_img is None and bool(s["ir_available"].item()) and len(s["ir_images"]) > 0:
                    ir_img = s["ir_images"][0]
            gas_vals = [s["gas"] for s in sessions if bool(s["gas_available"].item())]
            gas_avail = len(gas_vals) > 0
            gas_vec = torch.stack(gas_vals).mean(dim=0) if gas_avail else torch.zeros(6)
            day_entries.setdefault((fruit, label), []).append({
                "day_idx": day_idx, "rgb_img": rgb_img, "ir_img": ir_img,
                "gas_vec": gas_vec, "gas_avail": gas_avail, "fruit_idx": sessions[0]["fruit_idx"],
                "label": sessions[0]["label"], "days_until": sessions[0]["days_until_spoilage"], "fruit": fruit,
            })
        self.trajectories = []
        for (fruit, label), days in day_entries.items():
            days.sort(key=lambda d: d["day_idx"])
            last_observed_idx = None
            for d in days:
                d["delta"] = 0.0 if last_observed_idx is None else float(d["day_idx"] - last_observed_idx)
                if d["gas_avail"]:
                    last_observed_idx = d["day_idx"]
            self.trajectories.append({"fruit": fruit, "label": label, "days": days})
        print(f"  DayLevelSequenceDataset: {len(self.trajectories)} trajectories (split={split})")

    def __len__(self):
        return len(self.trajectories)

    def __getitem__(self, idx):
        traj = self.trajectories[idx]["days"]
        T = len(traj)
        blank_rgb = torch.zeros(3, IMAGE_SIZE, IMAGE_SIZE); blank_ir = torch.zeros(3, IMAGE_SIZE, IMAGE_SIZE)
        rgb_seq = torch.stack([d["rgb_img"] if d["rgb_img"] is not None else blank_rgb for d in traj])
        ir_seq = torch.stack([d["ir_img"] if d["ir_img"] is not None else blank_ir for d in traj])
        rgb_avail = torch.tensor([d["rgb_img"] is not None for d in traj], dtype=torch.bool)
        ir_avail = torch.tensor([d["ir_img"] is not None for d in traj], dtype=torch.bool)
        gas_seq = torch.stack([d["gas_vec"] for d in traj])
        gas_mask = torch.tensor([d["gas_avail"] for d in traj], dtype=torch.float32)
        gas_delta = torch.tensor([d["delta"] for d in traj], dtype=torch.float32)
        return {"rgb_seq": rgb_seq, "ir_seq": ir_seq, "rgb_avail": rgb_avail, "ir_avail": ir_avail,
                "gas_seq": gas_seq, "gas_mask": gas_mask, "gas_delta": gas_delta, "T": T,
                "fruit_idx": traj[0]["fruit_idx"], "label": traj[-1]["label"], "days_until": traj[-1]["days_until"],
                "fruit": self.trajectories[idx]["fruit"]}


def day_sequence_collate(batch):
    max_T = max(b["T"] for b in batch)
    B = len(batch)
    rgb = torch.zeros(B, max_T, 3, IMAGE_SIZE, IMAGE_SIZE); ir = torch.zeros(B, max_T, 3, IMAGE_SIZE, IMAGE_SIZE)
    rgb_avail = torch.zeros(B, max_T, dtype=torch.bool); ir_avail = torch.zeros(B, max_T, dtype=torch.bool)
    gas = torch.zeros(B, max_T, 6); gas_mask = torch.zeros(B, max_T); gas_delta = torch.zeros(B, max_T)
    pad_mask = torch.ones(B, max_T, dtype=torch.bool)
    fruit_idx = torch.zeros(B, dtype=torch.long); label = torch.zeros(B, dtype=torch.long); days_until = torch.zeros(B, dtype=torch.float32)
    fruits = []
    for i, b in enumerate(batch):
        T = b["T"]
        rgb[i, :T] = b["rgb_seq"]; ir[i, :T] = b["ir_seq"]
        rgb_avail[i, :T] = b["rgb_avail"]; ir_avail[i, :T] = b["ir_avail"]
        gas[i, :T] = b["gas_seq"]; gas_mask[i, :T] = b["gas_mask"]; gas_delta[i, :T] = b["gas_delta"]
        pad_mask[i, :T] = False
        fruit_idx[i] = b["fruit_idx"]; label[i] = b["label"]; days_until[i] = b["days_until"]
        fruits.append(b.get("fruit"))
    return {"rgb_seq": rgb, "ir_seq": ir, "rgb_avail": rgb_avail, "ir_avail": ir_avail,
            "gas_seq": gas, "gas_mask": gas_mask, "gas_delta": gas_delta, "pad_mask": pad_mask,
            "fruit_idx": fruit_idx, "label": label, "days_until": days_until, "fruit": fruits}


class OfflineAugmentedDayLevelSequenceDataset(Dataset):
    def __init__(self, base_dataset, transform_type="flip_rotation", n_copies=1):
        assert transform_type == "flip_rotation"
        self.base = base_dataset
        self.n_copies = n_copies
        self.index_map = []
        for traj_idx in range(len(base_dataset)):
            self.index_map.append((traj_idx, 0))
            for copy_id in range(1, n_copies + 1):
                self.index_map.append((traj_idx, copy_id))
        self.trajectories = [base_dataset.trajectories[traj_idx] for traj_idx, _ in self.index_map]
        print(f"OfflineAugmentedDayLevelSequenceDataset: {len(base_dataset)} -> {len(self)} trajectories")

    def __len__(self):
        return len(self.index_map)

    def _augment_image(self, img, seed):
        g = torch.Generator().manual_seed(seed)
        out = torch.flip(img, dims=[-1])
        angle = (torch.rand(1, generator=g).item() * 30.0) - 15.0
        return TF.rotate(out, angle)

    def __getitem__(self, idx):
        traj_idx, copy_id = self.index_map[idx]
        item = self.base[traj_idx]
        if copy_id == 0:
            return item
        seed_base = traj_idx * 10_000 + copy_id * 100
        item = dict(item)
        item["rgb_seq"] = torch.stack([self._augment_image(item["rgb_seq"][t], seed_base + t)
                                        for t in range(item["rgb_seq"].shape[0])])
        item["ir_seq"] = torch.stack([self._augment_image(item["ir_seq"][t], seed_base + t)
                                       for t in range(item["ir_seq"].shape[0])])
        return item


## Model

In [3]:
class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        hidden = max(channels // reduction, 8)
        self.mlp = nn.Sequential(nn.Conv2d(channels, hidden, 1, bias=False), nn.ReLU(inplace=True),
                                  nn.Conv2d(hidden, channels, 1, bias=False))
    def forward(self, x):
        return torch.sigmoid(self.mlp(self.avg_pool(x)) + self.mlp(self.max_pool(x)))


class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)
    def forward(self, x):
        avg_out = x.mean(dim=1, keepdim=True)
        max_out, _ = x.max(dim=1, keepdim=True)
        return torch.sigmoid(self.conv(torch.cat([avg_out, max_out], dim=1)))


class CBAM(nn.Module):
    def __init__(self, channels, reduction=16, spatial_kernel=7):
        super().__init__()
        self.channel_attn = ChannelAttention(channels, reduction)
        self.spatial_attn = SpatialAttention(spatial_kernel)
    def forward(self, x):
        x = x * self.channel_attn(x)
        sa = self.spatial_attn(x)
        return x * sa, sa


BACKBONE_FEATURE_DIMS = {"efficientnet_b0": 1280}


def _infer_feature_dim(backbone, img_size=IMAGE_SIZE):
    backbone.eval()
    with torch.no_grad():
        out = backbone(torch.zeros(1, 3, img_size, img_size))
    return out.shape[1]


def build_cnn_backbone(name):
    if name == "efficientnet_b0":
        base = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
        return base.features, BACKBONE_FEATURE_DIMS[name]
    elif name == "convnext_tiny":
        base = convnext_tiny(weights=ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
        backbone = base.features
        feat_dim = _infer_feature_dim(backbone)
        BACKBONE_FEATURE_DIMS[name] = feat_dim
        return backbone, feat_dim
    raise ValueError(f"Unknown backbone: {name}")


class VisualEncoderToggle(nn.Module):
    def __init__(self, backbone_name="efficientnet_b0", freeze_backbone=True, use_cbam=True, chunk_size=8):
        super().__init__()
        self.use_cbam = use_cbam
        self.backbone, self.feature_dim = build_cnn_backbone(backbone_name)
        self.chunk_size = chunk_size
        if use_cbam:
            self.cbam = CBAM(self.feature_dim)
        self.pool = nn.AdaptiveAvgPool2d(1)
        if freeze_backbone:
            for p in self.backbone.parameters():
                p.requires_grad = False

    def freeze_backbone(self):
        for p in self.backbone.parameters():
            p.requires_grad = False

    def unfreeze_backbone(self):
        for p in self.backbone.parameters():
            p.requires_grad = True

    def unfreeze_last_n_layers(self, n: int):
        self.freeze_backbone()
        for child in list(self.backbone.children())[-n:]:
            for p in child.parameters():
                p.requires_grad = True

    def forward(self, seq):
        B, T, C, H, W = seq.shape
        flat = seq.reshape(B * T, C, H, W)
        feats_list = []
        needs_ckpt = (torch.is_grad_enabled() and any(p.requires_grad for p in self.backbone.parameters()))
        for start in range(0, flat.shape[0], self.chunk_size):
            chunk = flat[start:start + self.chunk_size]
            feat_map = checkpoint(self.backbone, chunk, use_reentrant=False) if needs_ckpt else self.backbone(chunk)
            if self.use_cbam:
                feat_map, _ = self.cbam(feat_map)
            feats_list.append(self.pool(feat_map).flatten(1))
        return torch.cat(feats_list, dim=0).view(B, T, -1)


class GRUDCell(nn.Module):
    def __init__(self, input_dim, hidden_dim, x_mean=None):
        super().__init__()
        self.input_dim = input_dim; self.hidden_dim = hidden_dim
        if x_mean is None:
            x_mean = [0.0] * input_dim
        self.register_buffer("x_mean", torch.as_tensor(x_mean, dtype=torch.float32))
        self.W_gamma_x = nn.Linear(1, input_dim)
        self.W_gamma_h = nn.Linear(1, hidden_dim)
        self.gru_cell = nn.GRUCell(input_dim * 2, hidden_dim)
    def forward(self, x_t, m_t, delta_t, x_last, h_prev):
        gamma_x = torch.exp(-torch.clamp(self.W_gamma_x(delta_t), min=0.0))
        x_bar = self.x_mean.unsqueeze(0).expand_as(x_t)
        x_hat = m_t * x_t + (1 - m_t) * (gamma_x * x_last + (1 - gamma_x) * x_bar)
        gamma_h = torch.exp(-torch.clamp(self.W_gamma_h(delta_t), min=0.0))
        h_t = self.gru_cell(torch.cat([x_hat, m_t], dim=-1), gamma_h * h_prev)
        return h_t, x_hat


class GasGRUDSequenceEncoder(nn.Module):
    def __init__(self, input_dim=6, hidden_dim=64, x_mean=None):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.cell = GRUDCell(input_dim, hidden_dim, x_mean)
    def forward(self, gas_seq, gas_mask, gas_delta, pad_mask):
        B, T, D = gas_seq.shape
        device = gas_seq.device
        h = torch.zeros(B, self.hidden_dim, device=device)
        x_last = torch.zeros(B, D, device=device)
        h_seq = []
        for t in range(T):
            x_t = gas_seq[:, t]
            m_t = gas_mask[:, t].unsqueeze(-1).expand(-1, D)
            delta_t = gas_delta[:, t].unsqueeze(-1)
            valid_t = (~pad_mask[:, t]).float().unsqueeze(-1)
            h_new, x_hat = self.cell(x_t, m_t, delta_t, x_last, h)
            h = valid_t * h_new + (1 - valid_t) * h
            x_last = torch.where(m_t.bool(), x_t, x_last)
            h_seq.append(h)
        return torch.stack(h_seq, dim=1)


class CrossModalFusion(nn.Module):
    def __init__(self, d_model=64, num_heads=4, dropout=0.1):
        super().__init__()
        self.attn = nn.MultiheadAttention(embed_dim=d_model, num_heads=num_heads, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(d_model)
    def forward(self, rgb_t, ir_t, gas_t, rgb_avail_t, ir_avail_t, gas_avail_t):
        tokens = torch.stack([rgb_t, ir_t, gas_t], dim=1)
        avail = torch.stack([rgb_avail_t, ir_avail_t, gas_avail_t], dim=1)
        key_padding_mask = ~avail
        fully_missing = key_padding_mask.all(dim=1)
        if fully_missing.any():
            key_padding_mask = key_padding_mask.clone()
            key_padding_mask[fully_missing] = False
        attended, attn_w = self.attn(tokens, tokens, tokens, key_padding_mask=key_padding_mask,
                                      need_weights=True, average_attn_weights=True)
        out = self.norm(tokens + attended)
        return out.mean(dim=1), attn_w


class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))
    def forward(self, x):
        return x + self.pe[:, :x.shape[1]]


class TemporalTransformer(nn.Module):
    def __init__(self, d_model=64, nhead=4, num_layers=2, dim_feedforward=512, dropout=0.1, max_len=100):
        super().__init__()
        self.pos_enc = SinusoidalPositionalEncoding(d_model, max_len)
        layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
                                            dropout=dropout, batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers)
    def forward(self, z_seq, pad_mask):
        z_seq = self.pos_enc(z_seq)
        return self.encoder(z_seq, src_key_padding_mask=pad_mask)


def gather_last_valid(H, pad_mask):
    device = H.device
    lengths = (~pad_mask).sum(dim=1)
    last_idx = (lengths - 1).clamp(min=0)
    B = H.shape[0]
    return H[torch.arange(B, device=device), last_idx], last_idx


class UncertaintyWeightedLoss(nn.Module):
    def __init__(self, n_tasks: int = 3):
        super().__init__()
        self.log_sigma = nn.Parameter(torch.zeros(n_tasks))
    def forward(self, losses: list):
        total = 0.0
        for i, L_i in enumerate(losses):
            precision = torch.exp(-2 * self.log_sigma[i])
            total = total + 0.5 * precision * L_i + self.log_sigma[i]
        return total


class TrimodalFusionModelRegVariant(nn.Module):
    def __init__(self, d_model=64, gas_hidden=64, num_heads=4, temporal_layers=2,
                 dropout=0.5, cls_dropout=0.5, freeze_visual_backbone=True, reg_output_nonneg=True):
        super().__init__()
        self.d_model = d_model
        self.reg_output_nonneg = reg_output_nonneg
        self.rgb_encoder = VisualEncoderToggle("convnext_tiny", freeze_visual_backbone, use_cbam=True)
        self.ir_encoder = VisualEncoderToggle("efficientnet_b0", freeze_visual_backbone, use_cbam=True)
        self.gas_encoder = GasGRUDSequenceEncoder(input_dim=6, hidden_dim=gas_hidden)
        self.rgb_proj = nn.Sequential(nn.Linear(self.rgb_encoder.feature_dim, d_model), nn.LayerNorm(d_model), nn.ReLU(), nn.Dropout(dropout))
        self.ir_proj = nn.Sequential(nn.Linear(self.ir_encoder.feature_dim, d_model), nn.LayerNorm(d_model), nn.ReLU(), nn.Dropout(dropout))
        self.gas_proj = nn.Sequential(nn.Linear(gas_hidden, d_model), nn.LayerNorm(d_model), nn.ReLU(), nn.Dropout(dropout))
        self.fusion = CrossModalFusion(d_model=d_model, num_heads=num_heads, dropout=dropout)
        self.temporal = TemporalTransformer(d_model=d_model, nhead=num_heads, num_layers=temporal_layers, dropout=dropout)
        self.cls_head = nn.Sequential(nn.Linear(d_model, 128), nn.ReLU(), nn.Dropout(cls_dropout), nn.Linear(128, 2))
        reg_layers = [nn.Linear(d_model, 128), nn.ReLU(), nn.Dropout(cls_dropout), nn.Linear(128, 1)]
        if reg_output_nonneg:
            reg_layers.append(nn.ReLU())
        self.reg_head = nn.Sequential(*reg_layers)
        self.gas_recon_head = nn.Sequential(nn.Linear(d_model * 2, 128), nn.ReLU(), nn.Dropout(cls_dropout), nn.Linear(128, 6))

    def unfreeze_visual_last_n(self, n: int):
        self.rgb_encoder.unfreeze_last_n_layers(n); self.ir_encoder.unfreeze_last_n_layers(n)

    def unfreeze_visual_full(self):
        self.rgb_encoder.unfreeze_backbone(); self.ir_encoder.unfreeze_backbone()

    def forward(self, batch):
        device = next(self.parameters()).device
        rgb_seq = batch["rgb_seq"].to(device); ir_seq = batch["ir_seq"].to(device)
        rgb_avail = batch["rgb_avail"].to(device); ir_avail = batch["ir_avail"].to(device)
        gas_seq = batch["gas_seq"].to(device); gas_mask = batch["gas_mask"].to(device)
        gas_delta = batch["gas_delta"].to(device); pad_mask = batch["pad_mask"].to(device)
        B, T = rgb_avail.shape
        rgb_feat = self.rgb_encoder(rgb_seq); ir_feat = self.ir_encoder(ir_seq)
        gas_feat = self.gas_encoder(gas_seq, gas_mask, gas_delta, pad_mask)
        rgb_proj = self.rgb_proj(rgb_feat) * rgb_avail.unsqueeze(-1).float()
        ir_proj = self.ir_proj(ir_feat) * ir_avail.unsqueeze(-1).float()
        gas_proj = self.gas_proj(gas_feat) * gas_mask.unsqueeze(-1)
        rgb_flat = rgb_proj.reshape(B * T, -1); ir_flat = ir_proj.reshape(B * T, -1); gas_flat = gas_proj.reshape(B * T, -1)
        rgb_av_flat = rgb_avail.reshape(B * T); ir_av_flat = ir_avail.reshape(B * T); gas_av_flat = gas_mask.reshape(B * T).bool()
        z_flat, _ = self.fusion(rgb_flat, ir_flat, gas_flat, rgb_av_flat, ir_av_flat, gas_av_flat)
        z_seq = z_flat.reshape(B, T, -1)
        H = self.temporal(z_seq, pad_mask)
        H_T, last_idx = gather_last_valid(H, pad_mask)
        cls_logits = self.cls_head(H_T); reg_output = self.reg_head(H_T)
        rgb_last = rgb_proj[torch.arange(B, device=device), last_idx]
        ir_last = ir_proj[torch.arange(B, device=device), last_idx]
        gas_recon = self.gas_recon_head(torch.cat([rgb_last, ir_last], dim=1))
        return {"cls_logits": cls_logits, "reg_output": reg_output, "gas_recon": gas_recon, "last_idx": last_idx}

## Target transforms: Z-score (per fruit), Fraction (per fruit), Log (global)

In [4]:
class ZScoreTransform:
    def __init__(self, stats):
        self.stats = stats
    def to_norm(self, fruit, raw):
        s = self.stats[fruit]
        return (raw - s["mean"]) / s["std"]
    def from_norm(self, fruit, norm):
        s = self.stats[fruit]
        return norm * s["std"] + s["mean"]


class FractionTransform:
    def __init__(self, stats):
        self.stats = stats
    def to_norm(self, fruit, raw):
        return raw / self.stats[fruit]["max"]
    def from_norm(self, fruit, norm):
        return norm * self.stats[fruit]["max"]


class LogTransform:
    def to_norm(self, fruit, raw):
        return math.log1p(raw)
    def from_norm(self, fruit, norm):
        return math.expm1(norm)


def build_per_fruit_stats(train_trajectories):
    per_fruit_days = defaultdict(list)
    for t in train_trajectories:
        raw_val = t["days"][-1]["days_until"]
        raw_val = raw_val.item() if hasattr(raw_val, "item") else float(raw_val)
        per_fruit_days[t["fruit"]].append(raw_val)

    zscore_stats, frac_stats = {}, {}
    for fruit, values in per_fruit_days.items():
        arr = np.array(values, dtype=np.float64)
        mean_v = float(arr.mean())
        std_v = float(arr.std())
        if std_v < 1e-6:
            std_v = 1.0
        zscore_stats[fruit] = {"mean": mean_v, "std": std_v, "n": len(values)}
        max_v = float(arr.max())
        if max_v < 1e-6:
            max_v = 1.0
        frac_stats[fruit] = {"max": max_v, "n": len(values)}
    return zscore_stats, frac_stats, dict(per_fruit_days)


## Training / evaluation helpers

In [5]:
def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)


def build_weighted_sampler(dataset):
    labels = [int(t["label"]) for t in dataset.trajectories]
    n_total = len(labels); n_spoiled = max(sum(labels), 1); n_fresh = max(n_total - sum(labels), 1)
    w_spoiled = n_total / (2 * n_spoiled); w_fresh = n_total / (2 * n_fresh)
    weights = [w_spoiled if l == 1 else w_fresh for l in labels]
    return WeightedRandomSampler(weights=weights, num_samples=n_total, replacement=True)


def calibrate_threshold(scores, labels, thresholds=None):
    if thresholds is None:
        thresholds = THRESHOLDS_TO_SWEEP
    if len(set(labels)) < 2:
        return 0.5, 0.0
    best_t, best_f1 = 0.5, 0.0
    for t in thresholds:
        preds = [1 if s >= t else 0 for s in scores]
        f = f1_score(labels, preds, average="binary", zero_division=0)
        if f > best_f1:
            best_f1, best_t = f, t
    return best_t, best_f1


def compute_loss(out, batch, uncertainty_loss, cfg, device, target_transform):
    labels = batch["label"].to(device)
    fruits = batch["fruit"]
    raw_days = batch["days_until"].to(device)
    norm_days = torch.tensor([target_transform.to_norm(f, d.item()) for f, d in zip(fruits, raw_days)],
                              device=device, dtype=torch.float32)
    class_weights = torch.tensor(cfg["cls_class_weights"], device=device)
    cls_loss = nn.CrossEntropyLoss(weight=class_weights)(out["cls_logits"], labels)
    reg_loss = nn.SmoothL1Loss()(out["reg_output"].squeeze(1), norm_days)
    pad_mask = batch["pad_mask"].to(device); gas_mask = batch["gas_mask"].to(device); gas_seq = batch["gas_seq"].to(device)
    last_idx = out["last_idx"]; B = labels.shape[0]
    gas_avail_last = gas_mask[torch.arange(B, device=device), last_idx].bool()
    gas_true_last = gas_seq[torch.arange(B, device=device), last_idx]
    recon_loss = (nn.MSELoss()(out["gas_recon"][gas_avail_last], gas_true_last[gas_avail_last])
                  if gas_avail_last.any() else torch.zeros((), device=device))
    total_loss = uncertainty_loss([cls_loss, reg_loss, recon_loss])
    return total_loss


def run_epoch(model, uncertainty_loss, loader, optimizer, cfg, target_transform, train=True, threshold=None):
    model.train() if train else model.eval()
    uncertainty_loss.train() if train else uncertainty_loss.eval()
    device = cfg["device"]; total_loss = 0.0
    all_labels, all_probs = [], []
    all_days_gt_raw, all_days_pred_raw = [], []
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for batch in loader:
            out = model(batch)
            loss = compute_loss(out, batch, uncertainty_loss, cfg, device, target_transform)
            if train:
                optimizer.zero_grad(); loss.backward()
                nn.utils.clip_grad_norm_(list(model.parameters()) + list(uncertainty_loss.parameters()), max_norm=1.0)
                optimizer.step()
            total_loss += loss.item()
            probs = torch.softmax(out["cls_logits"], dim=1).detach().cpu().numpy()
            all_labels.extend(batch["label"].cpu().numpy().tolist())
            all_probs.extend(probs.tolist())
            raw_pred = [target_transform.from_norm(f, p) for f, p in
                        zip(batch["fruit"], out["reg_output"].squeeze(1).detach().cpu().numpy().tolist())]
            all_days_gt_raw.extend(batch["days_until"].cpu().numpy().tolist())
            all_days_pred_raw.extend(raw_pred)
    all_probs_arr = np.array(all_probs)
    if train:
        all_preds = all_probs_arr.argmax(axis=1).tolist(); threshold_used = 0.5
    else:
        if threshold is None:
            raise ValueError("run_epoch(train=False) requires a pre-calibrated threshold.")
        threshold_used = threshold
        all_preds = (all_probs_arr[:, 1] >= threshold_used).astype(int).tolist()
    f1 = f1_score(all_labels, all_preds, average="binary", zero_division=0)
    try:
        auc = roc_auc_score(all_labels, all_probs_arr[:, 1])
    except ValueError:
        auc = float("nan")
    mae = mean_absolute_error(all_days_gt_raw, all_days_pred_raw)
    return {"f1": round(f1, 4), "auc": round(auc, 4), "mae": round(mae, 4),
            "loss": round(total_loss / max(len(loader), 1), 4), "threshold": threshold_used}


def collect_probs_labels_days(model, loader, target_transform):
    model.eval()
    all_labels, all_probs = [], []
    all_days_gt_raw, all_days_pred_raw = [], []
    with torch.no_grad():
        for batch in loader:
            out = model(batch)
            probs = torch.softmax(out["cls_logits"], dim=1).cpu().numpy()
            all_labels.extend(batch["label"].cpu().numpy().tolist())
            all_probs.extend(probs.tolist())
            raw_pred = [target_transform.from_norm(f, p) for f, p in
                        zip(batch["fruit"], out["reg_output"].squeeze(1).cpu().numpy().tolist())]
            all_days_gt_raw.extend(batch["days_until"].cpu().numpy().tolist())
            all_days_pred_raw.extend(raw_pred)
    return np.array(all_probs), all_labels, all_days_gt_raw, all_days_pred_raw


def get_cfg():
    return {"epochs": 30, "batch_size": 4, "lr": 1e-4, "lr_finetune": 1e-5, "weight_decay": 1e-4,
            "cls_class_weights": [1.0, 2.3], "device": DEVICE}


cfg = get_cfg()
D_MODEL, DROPOUT = 64, 0.5

## Data

In [8]:
train_ds_base = DayLevelSequenceDataset(split="train")
val_ds = DayLevelSequenceDataset(split="val", shared_pixel_cache=train_ds_base._pixel_cache)
test_ds = DayLevelSequenceDataset(split="test", shared_pixel_cache=train_ds_base._pixel_cache)

train_aug = OfflineAugmentedDayLevelSequenceDataset(train_ds_base, "flip_rotation", n_copies=1)
train_loader = DataLoader(train_aug, batch_size=cfg["batch_size"], sampler=build_weighted_sampler(train_aug),
                           collate_fn=day_sequence_collate, num_workers=0)
train_loader_calib = DataLoader(train_ds_base, batch_size=cfg["batch_size"], shuffle=False,
                                 collate_fn=day_sequence_collate, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=cfg["batch_size"], shuffle=False,
                         collate_fn=day_sequence_collate, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, collate_fn=day_sequence_collate, num_workers=0)

trainval_trajectories = train_ds_base.trajectories + val_ds.trajectories
class _TrainValPool(Dataset):
    def __init__(self, trajectories): self.trajectories = trajectories
    def __len__(self): return len(self.trajectories)
    def __getitem__(self, idx):
        tmp = DayLevelSequenceDataset.__new__(DayLevelSequenceDataset)
        tmp.trajectories = self.trajectories
        return DayLevelSequenceDataset.__getitem__(tmp, idx)

trainval_pool = _TrainValPool(trainval_trajectories)
trainval_loader_calib = DataLoader(trainval_pool, batch_size=1, shuffle=False,
                                    collate_fn=day_sequence_collate, num_workers=0)

print(f"train (incl. augmentation): {len(train_aug)} | val: {len(val_ds)} | "
      f"train+val (test-calibration): {len(trainval_pool)} | test: {len(test_ds)}")

ZSCORE_STATS, FRAC_STATS, per_fruit_raw = build_per_fruit_stats(train_ds_base.trajectories)
zscore_transform = ZScoreTransform(ZSCORE_STATS)
frac_transform = FractionTransform(FRAC_STATS)
log_transform = LogTransform()

print("\nPer-fruit train stats (Z and F only)")
for fruit in sorted(per_fruit_raw.keys()):
    n = ZSCORE_STATS[fruit]["n"]
    zs = [round(zscore_transform.to_norm(fruit, v), 3) for v in per_fruit_raw[fruit]]
    print(f"  {fruit:<20} n={n}  raw={per_fruit_raw[fruit]}  z={zs}  "
          f"frac-max={FRAC_STATS[fruit]['max']:.2f}")

with open(OUT_DIR / "target_norm_stats.json", "w") as f:
    json.dump({"zscore": ZSCORE_STATS, "fraction": FRAC_STATS}, f, indent=2)


TrimodalDataset: 212 sessions (split=train)
  DayLevelSequenceDataset: 12 trajectories (split=train)
TrimodalDataset: 63 sessions (split=val)
  DayLevelSequenceDataset: 12 trajectories (split=val)
TrimodalDataset: 52 sessions (split=test)
  DayLevelSequenceDataset: 11 trajectories (split=test)
OfflineAugmentedDayLevelSequenceDataset: 12 -> 24 trajectories
train (incl. augmentation): 24 | val: 12 | train+val (test-calibration): 24 | test: 11

Per-fruit train stats (Z and F only)
  Banana               n=2  raw=[4.0, 0.0]  z=[1.0, -1.0]  frac-max=4.00
  Carrot               n=2  raw=[3.0, 0.0]  z=[1.0, -1.0]  frac-max=3.00
  Guava                n=2  raw=[4.0, 0.0]  z=[1.0, -1.0]  frac-max=4.00
  Indian_Gooseberry    n=2  raw=[7.0, 0.0]  z=[1.0, -1.0]  frac-max=7.00
  Mango                n=2  raw=[6.0, 0.0]  z=[1.0, -1.0]  frac-max=6.00
  Tomato               n=2  raw=[17.0, 0.0]  z=[1.0, -1.0]  frac-max=17.00


## Training: 5 seeds per variant (15 trainings total)

In [9]:
N_EPOCHS = cfg["epochs"]

VARIANTS = [
    ("variant_z_zscore", False, zscore_transform),
    ("variant_f_fraction", True, frac_transform),
    ("variant_log", True, log_transform),
]

def train_one_variant_seed(variant_name, reg_output_nonneg, target_transform, seed):
    seed_dir = OUT_DIR / variant_name / f"seed_{seed}"
    ckpt_path = seed_dir / "model_state.pt"
    hist_path = seed_dir / "history.json"

    if ckpt_path.exists() and hist_path.exists():
        print(f"  [{variant_name}] seed {seed}: already trained, skipping")
        return

    set_seed(seed)
    model = TrimodalFusionModelRegVariant(d_model=D_MODEL, dropout=DROPOUT, cls_dropout=DROPOUT,
                                           freeze_visual_backbone=True,
                                           reg_output_nonneg=reg_output_nonneg).to(cfg["device"])
    uncertainty_loss = UncertaintyWeightedLoss(n_tasks=3).to(cfg["device"])

    backbone_ids = {id(p) for p in list(model.rgb_encoder.backbone.parameters()) + list(model.ir_encoder.backbone.parameters())}
    other_params = [p for p in model.parameters() if id(p) not in backbone_ids]
    optimizer = AdamW([
        {"params": other_params, "lr": cfg["lr"]},
        {"params": list(model.rgb_encoder.backbone.parameters()) + list(model.ir_encoder.backbone.parameters()), "lr": cfg["lr_finetune"]},
        {"params": uncertainty_loss.parameters(), "lr": cfg["lr"]},
    ], weight_decay=cfg["weight_decay"])
    scheduler = CosineAnnealingLR(optimizer, T_max=N_EPOCHS)

    print(f"{'-' * 90}\n[{variant_name}] seed {seed}: training {N_EPOCHS} epochs\n{'-' * 90}")
    best_f1, best_state = -1.0, None
    history = []
    run_start = time.time()
    for epoch in range(1, N_EPOCHS + 1):
        if epoch == 6:
            model.unfreeze_visual_last_n(1)
        elif epoch == 8:
            model.unfreeze_visual_last_n(2)
        elif epoch == 12:
            model.unfreeze_visual_full()

        train_m = run_epoch(model, uncertainty_loss, train_loader, optimizer, cfg, target_transform, train=True)
        train_probs, train_labels, _, _ = collect_probs_labels_days(model, train_loader_calib, target_transform)
        thr, _ = calibrate_threshold(train_probs[:, 1].tolist(), train_labels)
        val_m = run_epoch(model, uncertainty_loss, val_loader, optimizer, cfg, target_transform, train=False, threshold=thr)
        scheduler.step()

        is_best = val_m["f1"] > best_f1
        if is_best:
            best_f1 = val_m["f1"]
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        history.append({"epoch": epoch, "train_f1": train_m["f1"], "val_f1": val_m["f1"],
                         "val_auc": val_m["auc"], "val_mae_raw": val_m["mae"], "val_threshold": thr})
        print(f"  [{variant_name}] seed {seed} | epoch {epoch:2d}/{N_EPOCHS} | train_F1={train_m['f1']:.3f} | "
              f"val_F1={val_m['f1']:.3f} val_AUC={val_m['auc']:.3f} val_MAE(raw)={val_m['mae']:.2f} "
              f"thr={thr:.2f}{'  *NEW BEST*' if is_best else ''}")

    total_time = time.time() - run_start
    seed_dir.mkdir(parents=True, exist_ok=True)
    torch.save(best_state, ckpt_path)
    with open(hist_path, "w") as f:
        json.dump(history, f, indent=2, default=str)
    print(f"  [{variant_name}] seed {seed}: DONE in {total_time / 60:.1f} min, best val F1={best_f1:.4f}")


TRAIN_FAILURES = []
for variant_name, reg_output_nonneg, transform in VARIANTS:
    for seed in range(N_SEEDS):
        try:
            train_one_variant_seed(variant_name, reg_output_nonneg, transform, seed)
        except Exception as e:
            tb = traceback.format_exc()
            print(f"  [FAIL] [{variant_name}] seed {seed}: {type(e).__name__}: {e}\n{tb}")
            TRAIN_FAILURES.append({"variant": variant_name, "seed": seed, "type": type(e).__name__,
                                    "message": str(e), "traceback": tb})
            FAILURES.append({"stage": "training", "variant": variant_name, "seed": seed,
                              "type": type(e).__name__, "message": str(e), "traceback": tb})

for variant_name, _, _ in VARIANTS:
    n_trained = sum(1 for s in range(N_SEEDS) if (OUT_DIR / variant_name / f"seed_{s}" / "model_state.pt").exists())
    print(f"{variant_name}: {n_trained} / {N_SEEDS} seeds trained.")
print(f"{len(TRAIN_FAILURES)} training failure(s).")


  [variant_z_zscore] seed 0: already trained, skipping


/var/folders/1m/bg8_bp9s1dzgb1v2l_3t0kx80000gn/T/ipykernel_2147/1141998318.py:178: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers)


------------------------------------------------------------------------------------------
[variant_z_zscore] seed 1: training 30 epochs
------------------------------------------------------------------------------------------
  [variant_z_zscore] seed 1 | epoch  1/30 | train_F1=0.828 | val_F1=0.706 val_AUC=0.694 val_MAE(raw)=1.92 thr=0.58  *NEW BEST*
  [variant_z_zscore] seed 1 | epoch  2/30 | train_F1=0.571 | val_F1=0.667 val_AUC=0.722 val_MAE(raw)=2.01 thr=0.10
  [variant_z_zscore] seed 1 | epoch  3/30 | train_F1=0.667 | val_F1=0.706 val_AUC=0.639 val_MAE(raw)=1.94 thr=0.61
  [variant_z_zscore] seed 1 | epoch  4/30 | train_F1=0.500 | val_F1=0.706 val_AUC=0.583 val_MAE(raw)=2.00 thr=0.54
  [variant_z_zscore] seed 1 | epoch  5/30 | train_F1=0.467 | val_F1=0.750 val_AUC=0.778 val_MAE(raw)=2.04 thr=0.58  *NEW BEST*
  [variant_z_zscore] seed 1 | epoch  6/30 | train_F1=0.688 | val_F1=0.706 val_AUC=0.722 val_MAE(raw)=2.08 thr=0.61
  [variant_z_zscore] seed 1 | epoch  7/30 | train_F1=0.552

/var/folders/1m/bg8_bp9s1dzgb1v2l_3t0kx80000gn/T/ipykernel_2147/1141998318.py:178: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers)


------------------------------------------------------------------------------------------
[variant_z_zscore] seed 2: training 30 epochs
------------------------------------------------------------------------------------------
  [variant_z_zscore] seed 2 | epoch  1/30 | train_F1=0.421 | val_F1=0.625 val_AUC=0.694 val_MAE(raw)=2.28 thr=0.38  *NEW BEST*
  [variant_z_zscore] seed 2 | epoch  2/30 | train_F1=0.333 | val_F1=0.727 val_AUC=0.611 val_MAE(raw)=2.20 thr=0.46  *NEW BEST*
  [variant_z_zscore] seed 2 | epoch  3/30 | train_F1=0.375 | val_F1=0.533 val_AUC=0.472 val_MAE(raw)=2.12 thr=0.45
  [variant_z_zscore] seed 2 | epoch  4/30 | train_F1=0.667 | val_F1=0.706 val_AUC=0.778 val_MAE(raw)=2.23 thr=0.44
  [variant_z_zscore] seed 2 | epoch  5/30 | train_F1=0.667 | val_F1=0.600 val_AUC=0.583 val_MAE(raw)=2.23 thr=0.51
  [variant_z_zscore] seed 2 | epoch  6/30 | train_F1=0.545 | val_F1=0.667 val_AUC=0.722 val_MAE(raw)=2.29 thr=0.45
  [variant_z_zscore] seed 2 | epoch  7/30 | train_F1=0.476

...

------------------------------------------------------------------------------------------
[variant_log] seed 2: training 30 epochs
------------------------------------------------------------------------------------------
  [variant_log] seed 2 | epoch  1/30 | train_F1=0.421 | val_F1=0.667 val_AUC=0.722 val_MAE(raw)=2.17 thr=0.37  *NEW BEST*
  [variant_log] seed 2 | epoch  2/30 | train_F1=0.333 | val_F1=0.727 val_AUC=0.611 val_MAE(raw)=2.18 thr=0.47  *NEW BEST*
  [variant_log] seed 2 | epoch  3/30 | train_F1=0.375 | val_F1=0.588 val_AUC=0.556 val_MAE(raw)=2.18 thr=0.46
  [variant_log] seed 2 | epoch  4/30 | train_F1=0.720 | val_F1=0.750 val_AUC=0.806 val_MAE(raw)=2.16 thr=0.47  *NEW BEST*
  [variant_log] seed 2 | epoch  5/30 | train_F1=0.667 | val_F1=0.600 val_AUC=0.611 val_MAE(raw)=2.20 thr=0.53
  [variant_log] seed 2 | epoch  6/30 | train_F1=0.609 | val_F1=0.706 val_AUC=0.722 val_MAE(raw)=2.18 thr=0.48
  [variant_log] seed 2 | epoch  7/30 | train_F1=0.583 | val_F1=0.600 val_AUC=0.72

/var/folders/1m/bg8_bp9s1dzgb1v2l_3t0kx80000gn/T/ipykernel_2147/1141998318.py:178: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers)


------------------------------------------------------------------------------------------
[variant_log] seed 3: training 30 epochs
------------------------------------------------------------------------------------------
  [variant_log] seed 3 | epoch  1/30 | train_F1=0.381 | val_F1=0.667 val_AUC=0.444 val_MAE(raw)=2.17 thr=0.10  *NEW BEST*
  [variant_log] seed 3 | epoch  2/30 | train_F1=0.125 | val_F1=0.222 val_AUC=0.528 val_MAE(raw)=2.13 thr=0.47
  [variant_log] seed 3 | epoch  3/30 | train_F1=0.700 | val_F1=0.667 val_AUC=0.444 val_MAE(raw)=2.16 thr=0.10
  [variant_log] seed 3 | epoch  4/30 | train_F1=0.364 | val_F1=0.625 val_AUC=0.417 val_MAE(raw)=2.12 thr=0.49
  [variant_log] seed 3 | epoch  5/30 | train_F1=0.385 | val_F1=0.588 val_AUC=0.444 val_MAE(raw)=2.15 thr=0.51
  [variant_log] seed 3 | epoch  6/30 | train_F1=0.480 | val_F1=0.667 val_AUC=0.528 val_MAE(raw)=2.09 thr=0.10
  [variant_log] seed 3 | epoch  7/30 | train_F1=0.560 | val_F1=0.667 val_AUC=0.500 val_MAE(raw)=2.13 thr=

/var/folders/1m/bg8_bp9s1dzgb1v2l_3t0kx80000gn/T/ipykernel_2147/1141998318.py:178: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers)


------------------------------------------------------------------------------------------
[variant_log] seed 4: training 30 epochs
------------------------------------------------------------------------------------------
  [variant_log] seed 4 | epoch  1/30 | train_F1=0.286 | val_F1=0.667 val_AUC=0.611 val_MAE(raw)=2.17 thr=0.40  *NEW BEST*
  [variant_log] seed 4 | epoch  2/30 | train_F1=0.462 | val_F1=0.588 val_AUC=0.417 val_MAE(raw)=2.11 thr=0.40
  [variant_log] seed 4 | epoch  3/30 | train_F1=0.526 | val_F1=0.667 val_AUC=0.389 val_MAE(raw)=2.24 thr=0.10
  [variant_log] seed 4 | epoch  4/30 | train_F1=0.688 | val_F1=0.571 val_AUC=0.444 val_MAE(raw)=2.14 thr=0.44
  [variant_log] seed 4 | epoch  5/30 | train_F1=0.500 | val_F1=0.667 val_AUC=0.250 val_MAE(raw)=2.16 thr=0.10
  [variant_log] seed 4 | epoch  6/30 | train_F1=0.688 | val_F1=0.308 val_AUC=0.222 val_MAE(raw)=2.11 thr=0.52
  [variant_log] seed 4 | epoch  7/30 | train_F1=0.273 | val_F1=0.364 val_AUC=0.361 val_MAE(raw)=2.09 thr=

## Val ensemble comparison: Full Model vs. all three variants

In [13]:
def ensemble_val_metrics(variant_name, reg_output_nonneg, target_transform, n_seeds=N_SEEDS):
    seed_probs, seed_days_pred_raw, seed_thresholds = [], [], []
    val_labels_ref, val_days_gt_ref = None, None
    for seed in range(n_seeds):
        ckpt_path = OUT_DIR / variant_name / f"seed_{seed}" / "model_state.pt"
        if not ckpt_path.exists():
            continue
        model = TrimodalFusionModelRegVariant(d_model=D_MODEL, dropout=DROPOUT, cls_dropout=DROPOUT,
                                               freeze_visual_backbone=True,
                                               reg_output_nonneg=reg_output_nonneg).to(cfg["device"])
        model.load_state_dict(torch.load(ckpt_path, map_location=cfg["device"]))
        model.eval()

        val_probs, val_labels, val_days_gt, val_days_pred = collect_probs_labels_days(model, val_loader, target_transform)
        if val_labels_ref is None:
            val_labels_ref, val_days_gt_ref = val_labels, val_days_gt
        seed_probs.append(val_probs[:, 1])
        seed_days_pred_raw.append(val_days_pred)

        train_probs, train_labels, _, _ = collect_probs_labels_days(model, train_loader_calib, target_transform)
        thr, _ = calibrate_threshold(train_probs[:, 1].tolist(), train_labels)
        seed_thresholds.append(thr)
        del model

    ens_probs = np.mean(seed_probs, axis=0)
    ens_days_pred = np.mean(seed_days_pred_raw, axis=0)
    ens_thr = float(np.median(seed_thresholds))
    ens_preds = (ens_probs >= ens_thr).astype(int)
    f1 = f1_score(val_labels_ref, ens_preds, average="binary", zero_division=0)
    try:
        auc = roc_auc_score(val_labels_ref, ens_probs)
    except ValueError:
        auc = float("nan")
    mae = mean_absolute_error(val_days_gt_ref, ens_days_pred)
    rmse = mean_squared_error(val_days_gt_ref, ens_days_pred) ** 0.5
    r2 = r2_score(val_days_gt_ref, ens_days_pred)
    return {"n_seeds": len(seed_probs), "threshold": ens_thr, "val_f1": round(float(f1), 4),
            "val_auc": round(float(auc), 4), "val_mae": round(float(mae), 4),
            "val_rmse": round(float(rmse), 4), "val_r2": round(float(r2), 4)}


VAL_RESULTS = {}
for variant_name, reg_output_nonneg, transform in VARIANTS:
    VAL_RESULTS[variant_name] = ensemble_val_metrics(variant_name, reg_output_nonneg, transform)

print("  VAL ENSEMBLE COMPARISON")
print(f"{'':<25}{'F1':>10}{'AUC':>10}{'MAE(raw)':>12}{'RMSE(raw)':>12}{'R2(raw)':>10}")
print(f"{'Full Model (known)':<25}{FULL_MODEL_VAL_F1_KNOWN:>10.4f}{'--':>10}{'--':>12}{'--':>12}{'--':>10}")
for variant_name, label in [("variant_z_zscore", "Variant Z (z-score)"),
                             ("variant_f_fraction", "Variant F (fraction)"),
                             ("variant_log", "Variant Log")]:
    r = VAL_RESULTS[variant_name]
    print(f"{label:<25}{r['val_f1']:>10.4f}{r['val_auc']:>10.4f}{r['val_mae']:>12.4f}{r['val_rmse']:>12.4f}{r['val_r2']:>10.4f}")

with open(OUT_DIR / "val_comparison.json", "w") as f:
    json.dump({"full_model_val_f1_known": FULL_MODEL_VAL_F1_KNOWN, **VAL_RESULTS}, f, indent=2)


/var/folders/1m/bg8_bp9s1dzgb1v2l_3t0kx80000gn/T/ipykernel_2147/1141998318.py:178: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers)
/var/folders/1m/bg8_bp9s1dzgb1v2l_3t0kx80000gn/T/ipykernel_2147/1141998318.py:178: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers)
/var/folders/1m/bg8_bp9s1dzgb1v2l_3t0kx80000gn/T/ipykernel_2147/1141998318.py:178: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers)
/var/folders/1m/bg8_bp9s1dzgb1v2l_3t0kx80000gn/T/ipykernel_2147/1141998318.py:178: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_lay

  VAL ENSEMBLE COMPARISON
                                 F1       AUC    MAE(raw)   RMSE(raw)   R2(raw)
Full Model (known)           0.8000        --          --          --        --
Variant Z (z-score)          0.6667    0.7500      2.1311      2.9553   -0.1955
Variant F (fraction)         0.7500    0.6944      2.0832      2.8215   -0.0897
Variant Log                  0.0000    0.7500      2.1459      3.1917   -0.3944


# One-time Test Evaluation

In [15]:
def guarded_test_eval(variant_name, reg_output_nonneg, target_transform, n_seeds=N_SEEDS):
    result_path = OUT_DIR / variant_name / "test_evaluation.json"
    if result_path.exists():
        with open(result_path) as f:
            result = json.load(f)
        print(f"[GUARD] {variant_name} already evaluated on test once.")
        return result

    print(f"Running ONE-TIME test evaluation for {variant_name}...")
    seed_probs, seed_days_pred_raw, seed_thresholds = [], [], []
    test_labels_ref, test_days_gt_ref = None, None

    for seed in range(n_seeds):
        ckpt_path = OUT_DIR / variant_name / f"seed_{seed}" / "model_state.pt"
        if not ckpt_path.exists():
            print(f"  [SKIP] seed {seed}: no checkpoint")
            continue
        try:
            model = TrimodalFusionModelRegVariant(d_model=D_MODEL, dropout=DROPOUT, cls_dropout=DROPOUT,
                                                   freeze_visual_backbone=True,
                                                   reg_output_nonneg=reg_output_nonneg).to(cfg["device"])
            model.load_state_dict(torch.load(ckpt_path, map_location=cfg["device"]))
            model.eval()

            test_probs, test_labels, test_days_gt, test_days_pred = collect_probs_labels_days(model, test_loader, target_transform)
            if test_labels_ref is None:
                test_labels_ref, test_days_gt_ref = test_labels, test_days_gt
            else:
                assert test_labels == test_labels_ref, f"label order mismatch, seed {seed}"
            seed_probs.append(test_probs[:, 1])
            seed_days_pred_raw.append(test_days_pred)

            train_probs, train_labels, _, _ = collect_probs_labels_days(model, trainval_loader_calib, target_transform)
            thr, _ = calibrate_threshold(train_probs[:, 1].tolist(), train_labels)
            seed_thresholds.append(thr)
            del model
            print(f"  seed {seed}: collected, threshold={thr:.3f}")
        except Exception as e:
            tb = traceback.format_exc()
            print(f"  [FAIL] seed {seed}: {type(e).__name__}: {e}\n{tb}")
            FAILURES.append({"variant": variant_name, "seed": seed, "stage": "test_eval",
                              "type": type(e).__name__, "message": str(e), "traceback": tb})

    if not seed_probs:
        print(f"[FAIL] No checkpoints for {variant_name}")
        return None

    ens_probs = np.mean(seed_probs, axis=0)
    ens_days_pred = np.mean(seed_days_pred_raw, axis=0)
    ens_thr = float(np.median(seed_thresholds))
    ens_preds = (ens_probs >= ens_thr).astype(int)
    f1 = f1_score(test_labels_ref, ens_preds, average="binary", zero_division=0)
    try:
        auc = roc_auc_score(test_labels_ref, ens_probs)
    except ValueError:
        auc = float("nan")
    mae = mean_absolute_error(test_days_gt_ref, ens_days_pred)
    rmse = mean_squared_error(test_days_gt_ref, ens_days_pred) ** 0.5
    r2 = r2_score(test_days_gt_ref, ens_days_pred)

    result = {"variant": variant_name, "n_seeds": len(seed_probs), "threshold": round(ens_thr, 4),
              "test_f1": round(float(f1), 4), "test_auc": round(float(auc), 4),
              "test_mae": round(float(mae), 4), "test_rmse": round(float(rmse), 4),
              "test_r2": round(float(r2), 4), "n_test_trajectories": int(len(test_labels_ref))}
    with open(result_path, "w") as f:
        json.dump(result, f, indent=2)
    print(f"Saved -> {result_path}")
    return result

TEST_RESULTS = {}
for variant_name, reg_output_nonneg, transform in VARIANTS:
    TEST_RESULTS[variant_name] = guarded_test_eval(variant_name, reg_output_nonneg, transform)


Running ONE-TIME test evaluation for variant_z_zscore...


/var/folders/1m/bg8_bp9s1dzgb1v2l_3t0kx80000gn/T/ipykernel_2147/1141998318.py:178: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers)


  seed 0: collected, threshold=0.660


/var/folders/1m/bg8_bp9s1dzgb1v2l_3t0kx80000gn/T/ipykernel_2147/1141998318.py:178: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers)


  seed 1: collected, threshold=0.580


...

  seed 2: collected, threshold=0.480


/var/folders/1m/bg8_bp9s1dzgb1v2l_3t0kx80000gn/T/ipykernel_2147/1141998318.py:178: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers)


  seed 3: collected, threshold=0.730


/var/folders/1m/bg8_bp9s1dzgb1v2l_3t0kx80000gn/T/ipykernel_2147/1141998318.py:178: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers)


  seed 4: collected, threshold=0.510
Saved -> /Users/sjanani2073/Desktop/convscript/runs/13_regression_target_triage/variant_log/test_evaluation.json


## Summary

In [16]:
print("  FINAL COMPARISON: Full Model vs. Variant Z vs. Variant F vs. Variant Log")
print(f"\n{'Metric':<12}{'Full Model':>14}{'Variant Z':>14}{'Variant F':>14}{'Variant Log':>14}")
for metric_key, label in [("test_f1", "F1"), ("test_auc", "AUC"), ("test_mae", "MAE"),
                           ("test_rmse", "RMSE"), ("test_r2", "R2")]:
    full_val = FULL_MODEL_TEST_RESULT[metric_key]
    row = [full_val]
    for variant_name in ["variant_z_zscore", "variant_f_fraction", "variant_log"]:
        r = TEST_RESULTS[variant_name]
        row.append(r[metric_key] if r else float("nan"))
    print(f"{label:<12}{row[0]:>14.4f}{row[1]:>14.4f}{row[2]:>14.4f}{row[3]:>14.4f}")

any_promising = False
for variant_name, label in [("variant_z_zscore", "Variant Z"), ("variant_f_fraction", "Variant F"),
                             ("variant_log", "Variant Log")]:
    r = TEST_RESULTS[variant_name]
    if r and r["test_r2"] > FULL_MODEL_TEST_RESULT["test_r2"] and r["test_f1"] >= FULL_MODEL_TEST_RESULT["test_f1"] - 0.05:
        print(f"\n{label} improves R2 without a large classification cost")
        any_promising = True

if not any_promising:
    print("\nAll three target-engineering variants failed.")

with open(OUT_DIR / "final_comparison.json", "w") as f:
    json.dump({"full_model": FULL_MODEL_TEST_RESULT, **{k: v for k, v in TEST_RESULTS.items()},
               "val_results": VAL_RESULTS}, f, indent=2)

if FAILURES:
    print(f"\n{len(FAILURES)} failure(s) logged")
print(f"\nAll outputs saved under: {OUT_DIR}")


  FINAL COMPARISON: Full Model vs. Variant Z vs. Variant F vs. Variant Log

Metric          Full Model     Variant Z     Variant F   Variant Log
F1                  0.8000        0.7143        0.5714        0.0000
AUC                 0.7333        0.6000        0.5000        0.4000
MAE                 0.9188        2.6722        1.0559        0.9453
RMSE                1.0593        3.6691        1.2690        1.1671
R2                  0.0439      -10.4712       -0.3722       -0.1607

All three target-engineering variants failed.

All outputs saved under: /Users/sjanani2073/Desktop/convscript/runs/13_regression_target_triage
